In [2]:
# noqa
from dbase.database.db_utils import set_environment_context
set_environment_context("long_bbands")
from EventDriven.riskmanager.utils import populate_cache_with_chain
from algo.strategies.init_backtest import get_strategy_backtest_setup
from trade import set_pool_enabled, SIGNALS_TO_RUN
from trade.helpers.pools import _change_global_stream_level
from trade.helpers.helper import parse_option_tick
from module_test.raw_code.DataManagers.DataManagers import set_use_quotes
from algo.strategies.init_orders.utils import generate_trades_data
from algo.positions.vars import get_orders_table
from algo.strategies.init_orders.core import OpenOrdersPipeline
from trade.datamanager import BaseDataManager
from trade.datamanager._enums import OptionPricingModel, DivType, OptionSpotEndpointSource
from trade.datamanager.utils.logging import (
    change_all_optionlib_loggers_level,
    change_datamanager_factor_loggers_level,
    change_logging_in_all_datamanager_loggers,
)
from algo.strategies.utils.order_optimizer import FillOptimizer, Side
from EventDriven.dataclasses.orders import OrderRequest
import numpy as np
import pandas as pd
from EventDriven.riskmanager.picker import filter_contracts, _order_formatting, create_trade_id
from EventDriven.types import ResultsEnum, Order

2026-02-11 21:39:22 [long_bbands] algo.__init__ CRITICAL: ALGO_DIR not on main branch; skipping runtime safeguards.


Loading BokehJS ...

[get_engine] Creating engine for DB: master_config (base: master_config), PID: 18104
[get_engine] Creating engine for DB: portfolio_data_long_bbands (base: portfolio_data), PID: 18104


In [3]:
BaseDataManager.CONFIG.option_spot_endpoint_source = OptionSpotEndpointSource.QUOTE
bkt = get_strategy_backtest_setup("long_bbands", reset=True)
bkt.portfolio.is_backtest = False
rm = bkt.risk_manager


2026-02-11 21:39:33 [long_bbands] EventDriven._vars CRITICAL: USE_TEMP_CACHE set to: True. This will use a temporary cache that is cleared on exit. Utilize reset_persistent_cache() to reset the persistent cache.
Using full database query for trades data.
[get_engine] Creating engine for DB: portfolio_signals_long_bbands (base: portfolio_signals), PID: 18104
2026-02-11 21:39:35 [long_bbands] EventDriven._vars CRITICAL: USE_TEMP_CACHE set to: True. This will use a temporary cache that is cleared on exit. Utilize reset_persistent_cache() to reset the persistent cache.
[get_engine] Creating engine for DB: portfolio_config_long_bbands (base: portfolio_config), PID: 18104
2026-02-11 21:39:37 [long_bbands] algo.strategies._config_utils INFO: Differences found in backtest_config.yaml: {"root['configs']['OrderSchemaConfigs_1']['min_total_price']": {'new_value': 0.5, 'old_value': 0.95}}
2026-02-11 21:39:37 [long_bbands] algo.strategies._config_utils INFO: Configuration differences detected for s

In [4]:
order_request1 = OrderRequest(
    date="2026-01-09",
    symbol="BA",
    option_type="c",
    max_close=1.0,
    tick_cash=300.0,
    direction="LONG",
    signal_id="BA20260105LONG",
    spot=np.float64(234.52999877929688),
    chain_spot=np.float64(234.52999877929688),
    is_tick_cash_scaled=True,
)

order_request2 = OrderRequest(
    date="2026-01-16",
    symbol="AAPL",
    option_type="c",
    max_close=1.0,
    tick_cash=300.0,
    direction="LONG",
    signal_id="AAPL20260115LONG",
    spot=np.float64(255.52999877929688),
    chain_spot=np.float64(255.52999877929688),
    is_tick_cash_scaled=True,
)

In [5]:
## Testing order picker directly

order = rm.order_picker.get_order(request=order_request2)

2026-02-11 21:39:44 [long_bbands] QuantTools.EventDriven.riskmanager ERROR: Retrieved chain for AAPL on 2026-01-16


In [6]:
order.to_dict()

{'result': 'SUCCESSFUL',
 'signal_id': 'AAPL20260115LONG',
 'map_signal_id': 'AAPL20260115LONG',
 'date': datetime.date(2026, 1, 16),
 'data': {'trade_id': TradeID(&L:AAPL20260821C300&S:AAPL20260821C305),
  'long': ['AAPL20260821C300'],
  'short': ['AAPL20260821C305'],
  'close': np.float64(0.9500000000000002),
  'quantity': 1},
 'metrics': {'spread_pct_ratio': np.float64(0.21052631578947384),
  'spread_oi': np.float64(4117.0)}}

In [4]:
chain = populate_cache_with_chain(
    tick="AAPL",
    date="2026-01-16",
    chain_spot=255.52999877929688,
)

2026-02-11 20:18:44 [long_bbands] QuantTools.EventDriven.riskmanager ERROR: Retrieved chain for AAPL on 2026-01-16


In [5]:
chain

,datetime,root,expiration,strike,right,bid_size,closebid,ask_size,closeask,date,midpoint,weighted_midpoint,open_interest,opttick,chain_id,dte,spot,moneyness,pct_spread
0,2026-01-16,AAPL,2028-03-17,200.0,C,10,82.75,49,83.65,20260116,83.200,83.497458,261.0,AAPL20280317C200,AAPL20280317C200_2026-01-16,791,255.529999,0.782687,0.010817
1,2026-01-16,AAPL,2028-03-17,200.0,P,50,13.55,26,13.85,20260116,13.700,13.652632,946.0,AAPL20280317P200,AAPL20280317P200_2026-01-16,791,255.529999,1.277650,0.021898
2,2026-01-16,AAPL,2026-07-17,210.0,P,2,4.10,5,4.20,20260116,4.150,4.171429,198.0,AAPL20260717P210,AAPL20260717P210_2026-01-16,182,255.529999,1.216810,0.024096
3,2026-01-16,AAPL,2026-07-17,210.0,C,28,52.95,27,53.65,20260116,53.300,53.293636,76.0,AAPL20260717C210,AAPL20260717C210_2026-01-16,182,255.529999,0.821821,0.013133
4,2026-01-16,AAPL,2026-08-21,210.0,P,9,5.25,5,5.35,20260116,5.300,5.285714,873.0,AAPL20260821P210,AAPL20260821P210_2026-01-16,217,255.529999,1.216810,0.018868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3029,2026-01-16,AAPL,2028-01-21,200.0,P,81,12.85,33,13.15,20260116,13.000,12.936842,1147.0,AAPL20280121P200,AAPL20280121P200_2026-01-16,735,255.529999,1.277650,0.023077
3030,2026-01-16,AAPL,2026-05-15,210.0,P,28,2.44,31,2.62,20260116,2.530,2.534576,5242.0,AAPL20260515P210,AAPL20260515P210_2026-01-16,119,255.529999,1.216810,0.071146
3031,2026-01-16,AAPL,2026-05-15,210.0,C,10,50.20,41,50.65,20260116,50.425,50.561765,1140.0,AAPL20260515C210,AAPL20260515C210_2026-01-16,119,255.529999,0.821821,0.008924
3032,2026-01-16,AAPL,2026-06-18,210.0,P,46,3.20,41,3.50,20260116,3.350,3.341379,7764.0,AAPL20260618P210,AAPL20260618P210_2026-01-16,153,255.529999,1.216810,0.089552


In [64]:
schema_as_dict = rm.order_picker.get_order_schema(ticker="AAPL", option_type="c", )
schema_as_dict.data["min_moneyness"] = 0.5
schema_as_tuple = sorted(schema_as_dict.data.items())
schema_as_tuple

[('dte_tolerance', 60),
 ('increment', 0.25),
 ('max_attempts', 3),
 ('max_moneyness', 1.0),
 ('max_pct_width', 0.2),
 ('max_total_price', None),
 ('min_moneyness', 0.5),
 ('min_oi', 5),
 ('min_total_price', 0.95),
 ('option_type', 'c'),
 ('run_name', 'long_bbands'),
 ('spread_ticks', 1),
 ('strategy', 'vertical'),
 ('structure_direction', 'long'),
 ('target_dte', 270),
 ('tick', 'AAPL')]

In [7]:
rm.order_picker._order_resolution_config.__dict__

{'run_name': 'long_bbands',
 'resolve_enabled': True,
 'otm_moneyness_width': 0.45,
 'itm_moneyness_width': 0.1,
 'max_close': 10.0,
 'max_tries': 15,
 'max_dte_tolerance': 95}

In [8]:
filtered = filter_contracts(df=chain, schema=schema_as_dict, spot=255.52999877929688)
filtered

,datetime,root,expiration,strike,right,bid_size,closebid,ask_size,closeask,date,midpoint,weighted_midpoint,open_interest,opttick,chain_id,dte,spot,moneyness,pct_spread
0,2026-01-16,AAPL,2026-11-20,260.0,C,8,25.85,13,26.35,20260116,26.100,26.159524,133.0,AAPL20261120C260,AAPL20261120C260_2026-01-16,308,255.529999,0.982808,0.019157
1,2026-01-16,AAPL,2026-11-20,270.0,C,5,21.05,35,21.50,20260116,21.275,21.443750,59.0,AAPL20261120C270,AAPL20261120C270_2026-01-16,308,255.529999,0.946407,0.021152
2,2026-01-16,AAPL,2026-08-21,260.0,C,5,20.75,76,20.90,20260116,20.825,20.890741,1263.0,AAPL20260821C260,AAPL20260821C260_2026-01-16,217,255.529999,0.982808,0.007203
3,2026-01-16,AAPL,2026-09-18,260.0,C,10,22.35,80,22.50,20260116,22.425,22.483333,3464.0,AAPL20260918C260,AAPL20260918C260_2026-01-16,245,255.529999,0.982808,0.006689
4,2026-01-16,AAPL,2026-08-21,265.0,C,22,18.25,44,19.15,20260116,18.700,18.850000,106.0,AAPL20260821C265,AAPL20260821C265_2026-01-16,217,255.529999,0.964264,0.048128
5,2026-01-16,AAPL,2026-09-18,265.0,C,4,19.90,3,20.00,20260116,19.950,19.942857,2511.0,AAPL20260918C265,AAPL20260918C265_2026-01-16,245,255.529999,0.964264,0.005013
6,2026-01-16,AAPL,2026-08-21,270.0,C,5,16.00,5,16.15,20260116,16.075,16.075000,1761.0,AAPL20260821C270,AAPL20260821C270_2026-01-16,217,255.529999,0.946407,0.009331
7,2026-01-16,AAPL,2026-09-18,270.0,C,5,17.60,14,17.70,20260116,17.650,17.673684,1705.0,AAPL20260918C270,AAPL20260918C270_2026-01-16,245,255.529999,0.946407,0.005666
8,2026-01-16,AAPL,2026-08-21,275.0,C,8,13.95,18,14.10,20260116,14.025,14.053846,316.0,AAPL20260821C275,AAPL20260821C275_2026-01-16,217,255.529999,0.929200,0.010695
9,2026-01-16,AAPL,2026-09-18,275.0,C,8,15.50,34,15.60,20260116,15.550,15.580952,3110.0,AAPL20260918C275,AAPL20260918C275_2026-01-16,245,255.529999,0.929200,0.006431


## Vertical Spread

In [9]:
## build out a vertical spread chain. 




## Make into a grouper function
def vertical_spread_pairer_by_exp(
    row: pd.Series,
    spread_tick: int = 1,
    min_total_price: float = 0.5,
    max_total_price: float = 1.0,
) -> pd.DataFrame:
    """
    For a given row (option contract), find the corresponding leg of the vertical spread based on the spread_tick.
    Calculate spread metrics such as spread mid, spread bid, spread ask, bid-ask spread, spread percentage ratio, and combined open interest.
    Filter the resulting paired DataFrame based on the total spread mid price being within the specified min
    and max total price range.
    Args:
        row (pd.Series): A row from the sorted_chain DataFrame representing an option contract.
        spread_tick (int): The number of ticks between the legs of the spread.
        min_total_price (float): Minimum total price of the spread to filter the results.
        max_total_price (float): Maximum total price of the spread to filter the results.
    Returns:
        pd.DataFrame: A DataFrame containing the paired legs of the vertical spread and their calculated metrics, filtered by the total spread mid price.
    """
    tgt_details = ["opttick", "midpoint", "closebid", "closeask", "open_interest"]
    long_leg_details = row[tgt_details].reset_index(drop=True)
    short_leg_details = row[tgt_details].shift(-spread_tick).reset_index(drop=True)

    ## Drop last spread_tick rows which will have NaNs in the short leg details after the shift.
    valid_length = len(row) - spread_tick
    long_leg_details = long_leg_details.iloc[:valid_length]
    short_leg_details = short_leg_details.iloc[:valid_length]

    ## Produce relevant spread information.
    spread_bid = long_leg_details["closebid"] - short_leg_details["closeask"]
    spread_ask = long_leg_details["closeask"] - short_leg_details["closebid"]
    spread_mid = long_leg_details["midpoint"] - short_leg_details["midpoint"]
    bid_ask_spread = spread_ask - spread_bid
    spread_pct_ratio = abs(spread_bid - spread_ask) / spread_mid.replace(0, np.nan)  # Avoid division by zero.
    spread_oi = abs(long_leg_details["open_interest"] + short_leg_details["open_interest"])


    ## Combine into a DataFrame for analysis.
    paired_opttick = pd.concat((long_leg_details["opttick"], short_leg_details["opttick"], spread_mid, spread_bid, spread_ask,  bid_ask_spread, spread_pct_ratio, spread_oi), axis=1)
    paired_opttick.columns = ["long_leg_opttick", "short_leg_opttick", "spread_mid", "spread_bid", "spread_ask", "bid_ask_spread", "spread_pct_ratio", "spread_oi"]
    return paired_opttick[paired_opttick["spread_mid"].between(min_total_price, max_total_price)].reset_index(drop=True)


def _vertical_spread_pairer(
    filtered_chain: pd.DataFrame,
    schema: dict,
) -> pd.DataFrame:

    # Start by ordering by strike, from ITM to OTM.
    #   For calls, ITM is lower strike, for puts, ITM is higher strike.

    spread_tick = schema["spread_ticks"]
    is_call = schema["option_type"].lower() == "c"
    sorted_chain = filtered_chain.sort_values(
        by="strike",
        ascending=is_call,  # Calls: Ascending (lower strike = ITM). Puts: Descending (higher strike = ITM).
    ).reset_index(drop=True)

    # spread_ticks is the number of ticks between the legs of the spread.
    # For a call spread with spread_ticks=1, we buy the ITM call and sell the next lower strike call.
    # For a put spread with spread_ticks=1, we buy the ITM put and sell the next higher strike put.
    # For vertical spreads it is important that the legs are paired to expiration.
    first_exp = sorted_chain["expiration"].iloc[0]
    exp_sorted_chain = sorted_chain[sorted_chain["expiration"] == first_exp].reset_index(drop=True)
    pd.concat((exp_sorted_chain["opttick"], exp_sorted_chain["opttick"].shift(-spread_tick).reset_index(drop=True)), axis=1)

    vertical_chain = sorted_chain.groupby("expiration").apply(
        vertical_spread_pairer_by_exp,
        spread_tick=spread_tick,
        min_total_price=0.5,
        max_total_price=4.0,
    ).reset_index(level=1, drop=True).sort_index()
    
    
    ## Now we have our vertical spread chain with paired optticks and spread metrics for analysis.
    ## We pick the spread we want based on specific criteria. We sort based on (this is by priority):
    ## 1. spread_pct_ratio (we want this to be low, meaning the spread is relatively tight compared to its midpoint)
    ## 2. spread_oi (we want this to be high, meaning there's good liquidity in the spread)
    ## Finally, pick the top row as our chosen spread.
    vertical_chain.sort_values(by=["spread_pct_ratio", "spread_oi"], ascending=[True, False], inplace=True)
    picked_spread = vertical_chain.iloc[0] if not vertical_chain.empty else pd.Series() 


    return picked_spread

picked = _vertical_spread_pairer(
    filtered_chain=filtered,
    schema=schema_as_dict,
)
picked

long_leg_opttick     AAPL20260918C265
short_leg_opttick    AAPL20260918C270
spread_mid                        2.3
spread_bid                        2.2
spread_ask                        2.4
bid_ask_spread                    0.2
spread_pct_ratio             0.086957
spread_oi                      4216.0
Name: 2026-09-18 00:00:00, dtype: object

In [22]:

def _extract_order_for_vertical_spread(
        picked_spread:pd.Series
) -> dict:
    """
    
    """
    ## Extract order
    if not picked_spread.empty:
        long = [{"opttick": picked_spread["long_leg_opttick"]}]
        short = [{"opttick": picked_spread["short_leg_opttick"]}]
        leg_info = [("L", picked_spread["long_leg_opttick"]), ("S", picked_spread["short_leg_opttick"])]
        close_price = picked_spread["spread_mid"]
        trade_id = create_trade_id(legs = {"long": long, "short": short,})
        data = _order_formatting(
            trade_id=trade_id,
            legs=leg_info,
            close=close_price,
            dir=schema_as_dict["structure_direction"])
        order = {
            "result": ResultsEnum.SUCCESSFUL.value,
            "data": data,
        }
    else:
        order = {
            "result": ResultsEnum.UNSUCCESSFUL.value,
            "data": None,
        }
        
    return order

order = _extract_order_for_vertical_spread(picked)

In [27]:
def vertical_spread_order_builder(
        filtered_chain: pd.DataFrame,
        schema: dict,
) -> dict:
    """
    Build a vertical spread order based on the filtered option chain and the provided schema.
    Args:
        filtered_chain (pd.DataFrame): The filtered option chain DataFrame.
        schema (dict): The order schema containing parameters for building the vertical spread.
    Returns:
        dict: A dictionary containing the result status and order data.
    """
    picked_spread = _vertical_spread_pairer(
        filtered_chain=filtered_chain,
        schema=schema,
    )
    order = _extract_order_for_vertical_spread(picked_spread)
    return order
vertical_spread_order = vertical_spread_order_builder(
    filtered_chain=filtered,
    schema=schema_as_dict.data,
)
vertical_spread_order["data"]["trade_id"]

TradeID(&L:AAPL20260918C265&S:AAPL20260918C270)

## Naked Option

In [40]:
## build out a naked option chain. 




## Make into a grouper function
def naked_option_by_exp(
    row: pd.Series,
    min_total_price: float = 0.5,
    max_total_price: float = 1.0,
) -> pd.DataFrame:
    """
    For a given row (option contract), find the corresponding leg of the naked option based on the spread_tick.
    Calculate spread metrics such as spread mid, spread bid, spread ask, bid-ask spread, spread percentage ratio, and combined open interest.
    Filter the resulting paired DataFrame based on the total spread mid price being within the specified min
    and max total price range.
    Args:
        row (pd.Series): A row from the sorted_chain DataFrame representing an option contract.
        spread_tick (int): The number of ticks between the legs of the spread.
        min_total_price (float): Minimum total price of the spread to filter the results.
        max_total_price (float): Maximum total price of the spread to filter the results.
    Returns:
        pd.DataFrame: A DataFrame containing the paired legs of the vertical spread and their calculated metrics, filtered by the total spread mid price.
    """
    tgt_details = ["opttick", "midpoint", "closebid", "closeask", "open_interest"]
    long_leg_details = row[tgt_details].reset_index(drop=True)

    ## Produce relevant spread information.
    spread_bid = long_leg_details["closebid"]
    spread_ask = long_leg_details["closeask"]
    spread_mid = long_leg_details["midpoint"]
    bid_ask_spread = spread_ask - spread_bid
    spread_pct_ratio = abs(spread_bid - spread_ask) / spread_mid.replace(0, np.nan)  # Avoid division by zero.
    spread_oi = abs(long_leg_details["open_interest"])


    ## Combine into a DataFrame for analysis.
    paired_opttick = pd.concat((long_leg_details["opttick"], spread_mid, spread_bid, spread_ask,  bid_ask_spread, spread_pct_ratio, spread_oi), axis=1)
    paired_opttick.columns = ["long_leg_opttick", "spread_mid", "spread_bid", "spread_ask", "bid_ask_spread", "spread_pct_ratio", "spread_oi"]
    return paired_opttick[paired_opttick["spread_mid"].between(min_total_price, max_total_price)].reset_index(drop=True)


def _naked_option_finder(
    filtered_chain: pd.DataFrame,
    schema: dict,
) -> pd.DataFrame:
    """
    For a given filtered option chain, find the best option contract based on the spread_tick.
    Args:
        filtered_chain (pd.DataFrame): The filtered option chain DataFrame.
        schema (dict): The order schema containing parameters for building the naked option.
    Returns:
        pd.DataFrame: A Series containing the picked naked option contract details.
    """

    # Start by ordering by strike, from ITM to OTM.
    #   For calls, ITM is lower strike, for puts, ITM is higher strike.

    is_call = schema["option_type"].lower() == "c"
    sorted_chain = filtered_chain.sort_values(
        by="strike",
        ascending=is_call,  # Calls: Ascending (lower strike = ITM). Puts: Descending (higher strike = ITM).
    ).reset_index(drop=True)

    naked_option_chain = sorted_chain.groupby("expiration").apply(
        naked_option_by_exp,
        min_total_price=0.5,
        max_total_price=4.0,
    ).reset_index(level=1, drop=True).sort_index()
    
    ## Now we have our naked option chain with spread metrics for analysis.
    ## We pick the option we want based on specific criteria. We sort based on (this is by priority):
    ## 1. spread_pct_ratio (we want this to be low, meaning the spread is relatively tight compared to its midpoint)
    ## 2. spread_oi (we want this to be high, meaning there's good liquidity in the spread)
    ## Finally, pick the top row as our chosen spread.
    naked_option_chain.sort_values(by=["spread_pct_ratio", "spread_oi"], ascending=[True, False], inplace=True)
    picked_spread = naked_option_chain.iloc[0] if not naked_option_chain.empty else pd.Series() 


    return picked_spread

picked = _naked_option_finder(
    filtered_chain=filtered,
    schema=schema_as_dict,
)
picked

long_leg_opttick    AAPL20260918C335
spread_mid                     2.775
spread_bid                      2.75
spread_ask                       2.8
bid_ask_spread                  0.05
spread_pct_ratio            0.018018
spread_oi                      262.0
Name: 2026-09-18 00:00:00, dtype: object

In [ ]:

def _extract_order_for_naked_option(
        picked_spread:pd.Series,
        schema: dict,
) -> dict:
    """
    Extract order details for a naked option based on the picked spread and the provided schema.
    Args:
        picked_spread (pd.Series): A Series containing the details of the picked naked option contract.
        schema (dict): The order schema containing parameters for building the naked option.
    Returns:
        dict: A dictionary containing the result status and order data.
    """
    ## Extract order
    is_long = schema["structure_direction"].upper() == "LONG"
    if not picked_spread.empty:

        
        ## Determine leg info for order formatting
        long = [{"opttick": picked_spread["long_leg_opttick"]}] if is_long else []
        short = [{"opttick": picked_spread["short_leg_opttick"]}] if not is_long else []

        ## Determine leg info for order formatting
        leg_info = []
        if is_long:
            leg_info.append(("L", picked_spread["long_leg_opttick"]))
        else:
            leg_info.append(("S", picked_spread["long_leg_opttick"]))
        
        close_price = picked_spread["spread_mid"]
        trade_id = create_trade_id(legs = {"long": long, "short": short,})
        data = _order_formatting(
            trade_id=trade_id,
            legs=leg_info,
            close=close_price,
            dir=schema["structure_direction"])
        order = {
            "result": ResultsEnum.SUCCESSFUL.value,
            "data": data,
        }
    else:
        order = {
            "result": ResultsEnum.UNSUCCESSFUL.value,
            "data": None,
        }
        
    return order

order = _extract_order_for_naked_option(picked, schema=schema_as_dict)
order

{'result': 'SUCCESSFUL',
 'data': {'trade_id': TradeID(&L:AAPL20260918C335),
  'close': np.float64(2.775),
  'long': ['AAPL20260918C335'],
  'quantity': 1}}

In [38]:
def naked_option_order_builder(
        filtered_chain: pd.DataFrame,
        schema: dict,
) -> dict:
    """
    Build a vertical spread order based on the filtered option chain and the provided schema.
    Args:
        filtered_chain (pd.DataFrame): The filtered option chain DataFrame.
        schema (dict): The order schema containing parameters for building the vertical spread.
    Returns:
        dict: A dictionary containing the result status and order data.
    """
    picked_spread = _naked_option_finder(
        filtered_chain=filtered_chain,
        schema=schema,
    )
    order = _extract_order_for_naked_option(picked_spread, schema=schema)
    return order
naked_option_order = naked_option_order_builder(
    filtered_chain=filtered,
    schema=schema_as_dict.data,
)
naked_option_order["data"]["trade_id"]

TradeID(&L:AAPL20260918C335)

In [19]:
schema_as_dict.data

{'tick': 'AAPL',
 'target_dte': 270,
 'strategy': 'vertical',
 'structure_direction': 'long',
 'spread_ticks': 1,
 'dte_tolerance': 60,
 'min_moneyness': 1.0,
 'max_moneyness': 1.5,
 'min_total_price': 0.95,
 'option_type': 'c',
 'max_total_price': None,
 'max_attempts': 3,
 'increment': 0.25,
 'run_name': 'long_bbands',
 'max_pct_width': 0.2,
 'min_oi': 5}

In [ ]:
def validate_order(order:dict) -> bool:
    """
    Validates the order dictionary structure and contents.
    Raises ValueError if validation fails.
    """
    assert "result" in order, "Order must have a 'result' key."
    assert order["result"] in [e.value for e in ResultsEnum], f"Invalid result value: {order['result']}."
    if order["result"] == ResultsEnum.SUCCESSFUL.value:
        assert "data" in order, "Successful order must have 'data' key."
        data = order["data"]
        assert "trade_id" in data, "Order data must have 'trade_id'."
        assert "close" in data, "Order data must have 'close' price."
        assert "long" in data or "short" in data, "Order data must have at least 'long' or 'short' legs."
        if "long" in data:
            for leg in data["long"]:
                assert isinstance(leg, str), "Each long leg must be a string."
                try:
                    parse_option_tick(leg)
                except Exception as e:
                    raise ValueError(f"Invalid long leg opttick: {leg}. Error: {e}") from e
        if "short" in data:
            for leg in data["short"]:
                assert isinstance(leg, str), "Each short leg must be a string."
                try:
                    parse_option_tick(leg)
                except Exception as e:
                    raise ValueError(f"Invalid short leg opttick: {leg}. Error: {e}") from e
    return True

In [ ]:
validate_order(order)
order

{'result': 'SUCCESSFUL',
 'data': {'trade_id': '&L:AAPL20260918C275&S:AAPL20260918C285',
  'close': np.float64(3.650000000000002),
  'long': ['AAPL20260918C275'],
  'short': ['AAPL20260918C285'],
  'quantity': -1}}

In [ ]:
# Order.from_dict(order)

## Order builder factory's expected functions:
## 1. Generate Order
## 2. Extract Order
## 3. Validate Order
## Builder workflow/responsibilities:
## Workflow:
## 1. Recieve necessary parameters ie schema, tick, spot
## 2. Get chain, filter chain.
## 3. Look through generate order factory functions to find the one that matches the schema structure.
## 4. Generate actual order. It is expecting actual order
## 5. Validate order using the validate order function.
## Responsibilities:
## 1. Ensure the generated order adheres to the provided schema.
## 2. Ensure the generated order is valid and can be processed by downstream systems.

## TODO:
## 1. Consider killing the flipping of moneyness for calls vs puts. assume that the <1 is OTM for both calls and puts.
## The filter then takes into account the option type when applying the moneyness filter. This would simplify the schema and reduce potential confusion.

In [58]:
schema_as_tuple
schema_as_dict.data["max_total_price"] = 2

In [63]:
from EventDriven.riskmanager.picker.builder import order_builder
schema_as_dict.data["strategy"] = "naked"
order = order_builder(
    unfiltered_chain=chain,
    schema=schema_as_dict,
    spot=255.52999877929688,
)

order

{'result': 'SUCCESSFUL',
 'data': {'trade_id': TradeID(&L:AAPL20260821C340),
  'close': np.float64(1.885),
  'long': ['AAPL20260821C340'],
  'quantity': 1},
 'metrics': {'spread_pct_ratio': np.float64(0.026525198938991947),
  'spread_oi': np.float64(798.0)}}

In [123]:
order_request1.option_type = "C"
rm.order_picker._order_schema_config.strategy = "naked"

In [124]:
order_request1.chain_spot
# rm.order_picker.get_order_schema(ticker="AAPL", option_type="c").data

np.float64(234.52999877929688)

In [128]:
## Testing order picker directly

order = rm.order_picker.get_order(request=order_request2)

Calculating moneyness for puts: strike/spot
Calculating moneyness for puts: strike/spot
Calculating moneyness for puts: strike/spot


In [129]:
order.to_dict()

{'result': 'SUCCESSFUL',
 'signal_id': 'AAPL20260115LONG',
 'map_signal_id': 'AAPL20260115LONG',
 'date': datetime.date(2026, 1, 16),
 'data': {'trade_id': TradeID(&L:AAPL20260717P170),
  'long': ['AAPL20260717P170'],
  'short': [],
  'close': np.float64(0.98),
  'quantity': 1},
 'metrics': {'spread_pct_ratio': np.float64(0.04081632653061228),
  'spread_oi': np.float64(237.0)}}